# 03 — Results analysis and discussion framework

## Purpose

Use this notebook only after the planned experiments are complete. It provides prompts for evidence-based analysis and deliberately contains **no numerical results or conclusions**. Every statement should trace to saved run IDs, prediction tables, uncertainty estimates, and figures. Distinguish observations (what the measurements show) from explanations (why they may have occurred).

In [1]:
# Setup. Declares the exact completed runs this analysis reads. Every number below is
# copied from a saved metrics file or prediction table; nothing is recomputed with
# different conventions and nothing is entered by hand.
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
OUTPUT_ROOT = REPO_ROOT / "outputs"

METRIC_FILES = {
    "baseline": "test_metrics.json",
    "unseen_generator": "unseen_generator_metrics.json",
    "fine_tuning": "recovery_metrics.json",
    "ablation": "ablation_metrics.json",
}


def completed(prefix):
    runs = []
    for path in sorted(OUTPUT_ROOT.glob(f"{prefix}-*")):
        status = path / "status.json"
        if (status.is_file() and json.loads(status.read_text())["status"] == "completed"
                and (path / METRIC_FILES[prefix]).is_file()):
            runs.append(path)
    return runs


def load(prefix, run=None):
    """Return (run_dir, metrics_dict) for one experiment type, or (None, None)."""
    runs = completed(prefix)
    if not runs:
        return None, None
    chosen = run or runs[-1]
    return chosen, json.loads((chosen / METRIC_FILES[prefix]).read_text())


def provenance(run):
    """Print the identity every reported number must be traceable to."""
    config = (run / "resolved_config.yaml").read_text() if (run / "resolved_config.yaml").is_file() else ""
    print(f"run_id        : {run.name}")
    if "SMOKE" in config:
        print("WARNING       : SYNTHETIC SMOKE RUN - pipeline evidence only, NOT results")
    environment = run / "environment.json"
    if environment.is_file():
        env = json.loads(environment.read_text())
        packages = env.get("packages", {})
        print(f"environment   : python {env.get('python')} torch {packages.get('torch')} "
              f"transformers {packages.get('transformers')} cuda={env.get('cuda_available')}")


RUNS = {name: (completed(name)[-1] if completed(name) else None) for name in METRIC_FILES}
display(pd.DataFrame([
    {"experiment": name, "run": run.name if run else "NO COMPLETED RUN"}
    for name, run in RUNS.items()
]))

,experiment,run
0,baseline,NO COMPLETED RUN
1,unseen_generator,unseen_generator-20260808T144550443491Z-b386ea...
2,fine_tuning,fine_tuning-20260808T144640138049Z-6ec6cf50a2-...
3,ablation,NO COMPLETED RUN


## 1. Baseline

### Questions to answer

- Did the detector learn the in-distribution real/fake task under the declared known-generator setting?
- What do accuracy, precision, recall, F1, ROC-AUC, PR/average precision, and confusion counts each reveal?
- Are results stable across seeds, and do generator-wise values reveal hidden weaknesses?
- Do training curves support the selected checkpoint, or suggest underfitting/overfitting?
- Are there plausible dataset artefacts that could inflate baseline performance?

### Writing TODO

Describe the protocol and measured variation before interpretation. Treat this as the reference condition, not evidence of unseen-generator transfer. Insert only values regenerated from saved predictions.

In [2]:
# 1. Baseline - in-distribution reference.
run, metrics = load("baseline")
if run is None:
    print("No completed baseline run yet.")
else:
    provenance(run)
    overall = metrics["overall"]
    display(pd.DataFrame([{
        "roc_auc": overall["roc_auc"], "average_precision": overall["average_precision"],
        "accuracy": overall["accuracy"], "precision": overall["precision"],
        "recall": overall["recall"], "f1": overall["f1"],
        "threshold": overall["threshold"], "support": overall["support"],
        "best_epoch": metrics.get("best_epoch"),
    }]).T.rename(columns={0: "value"}))
    display(pd.DataFrame(metrics["per_generator"]).T[
        ["support", "accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
    ].sort_values("f1"))

No completed baseline run yet.


## 2. Unseen-generator performance

### Questions to answer

- How does performance change when the fake generator was absent from training and model selection?
- Is the change consistent across held-out generators and seeds, or generator-specific?
- Which error type changes most, and is that conclusion threshold-dependent?
- Do ROC/PR behaviour and score distributions suggest ranking failure, calibration/threshold shift, or both?
- Were baseline and unseen comparisons matched on model, training data policy, real pool, sample support, and metrics?

### Writing TODO

State the observed difference with uncertainty and actual sample counts. Do not attribute it to generator architecture without direct evidence; dataset source, content, compression, and preprocessing may also differ.

In [3]:
# 2. Unseen-generator performance.
#
# Both test sets are balanced 50/50 over the same fixed real pool, so precision, F1 and
# PR-AUC are comparable between them. Threshold-free metrics are listed FIRST: a
# fixed-threshold gap on an unseen generator largely measures calibration drift.
run, metrics = load("unseen_generator")
if run is None:
    print("No completed unseen-generator run yet.")
else:
    provenance(run)
    print(f"held out      : {metrics['held_out_generator']}")
    print(f"known         : {', '.join(metrics['known_generators'])}")
    print(f"seed          : {metrics['seed']}")
    print(f"manifest      : {metrics['manifest_sha256'][:32]}...")
    gap = metrics["generalisation_gap"]
    rows = []
    for label, key in (("roc_auc (threshold-free)", "roc_auc"),
                       ("average_precision (threshold-free)", "average_precision"),
                       ("f1 @ default threshold", "f1_at_default_threshold")):
        entry = gap.get(key, {})
        rows.append({"metric": label, "in_distribution": entry.get("in_distribution"),
                     "unseen": entry.get("unseen"), "absolute_drop": entry.get("absolute_drop")})
    selected = gap.get("f1_at_baseline_validation_selected_threshold", {})
    if "unseen" in selected:
        rows.append({"metric": "f1 @ validation-selected threshold",
                     "in_distribution": None, "unseen": selected["unseen"],
                     "absolute_drop": None})
    display(pd.DataFrame(rows))

    print("\ntest composition and thresholds (provenance):")
    display(pd.DataFrame([metrics["final_test_composition"]]).T.rename(columns={0: "value"}))
    display(pd.DataFrame(metrics["thresholds"]).T)

run_id        : unseen_generator-20260808T144550443491Z-b386ea5c10-bb06
environment   : python 3.11.15 torch 2.13.0 transformers 5.14.1 cuda=False
held out      : biggan
known         : midjourney, stable_diffusion_v1_4, adm
seed          : 42
manifest      : a8fe6d566b50f43563089d7d294aed2d...


,metric,in_distribution,unseen,absolute_drop
0,roc_auc (threshold-free),1.0,0.986111,0.013889
1,average_precision (threshold-free),1.0,0.988095,0.011905
2,f1 @ default threshold,1.0,0.000000,1.000000
3,f1 @ validation-selected threshold,NaN,0.956522,NaN



test composition and thresholds (provenance):


,value
final_test_sha256,081720061bef91805a556f9f441b70159722cdb90bd05a...
held_out_fake_count,12
policy,balanced_50_50_fixed_real_pool
positive_prevalence,0.5
real_count,12
real_pool_available,20
real_pool_sha256,5a07c7291a5ee246bb72bab882ef33f9a04358eb7e60e7...
real_test_pool_seed,20260808


,held_out_samples_used,provenance,selection_metric,selection_sample_count,selection_score,value
baseline_validation_selected,0,selected_on_seen_generator_validation_only__gr...,f1,56,1.0,0.31
default,NaN,fixed_prior_from_config_model.decision_threshold,NaN,NaN,NaN,0.5


## 3. Fine-tuning recovery

### Questions to answer

- Relative to the measured 0% condition, what changes at 5%, 10%, 20%, and 50%?
- What actual labelled sample counts do those percentages represent?
- Is improvement monotonic within uncertainty, and how sensitive is it to subset composition/training seed?
- Is recovery visible across threshold-dependent and ranking metrics, or only one family?
- Does recovery plateau, remain incomplete, or introduce a precision/recall trade-off?

### Writing TODO

Report distributions/intervals across declared seeds rather than a selected best run. Describe 'recovery' using a predeclared calculation and avoid implying performance at untested percentages.

In [4]:
# 3. Fine-tuning recovery.
#
# labelled_images_consumed counts EVERY labelled held-out image the cell used, including
# the adaptation-validation images spent on model selection and threshold selection.
run, metrics = load("fine_tuning")
if run is None:
    print("No completed fine-tuning run yet.")
else:
    provenance(run)
    print(f"held out          : {metrics['held_out_generator']}")
    print(f"adaptation pool   : {metrics['adaptation_pool_size']}")
    print(f"final test        : {metrics['final_unseen_test_size']} "
          f"(prevalence {metrics['final_test_composition']['positive_prevalence']})")
    print(f"baseline threshold: {metrics['baseline_threshold']} "
          f"({metrics['baseline_threshold_provenance']})")
    print(f"starting ckpt     : {metrics['starting_checkpoint_compatibility'].get('starting_checkpoint_metadata', {})}")

    cells = pd.read_csv(run / "recovery_cells.csv")
    columns = [c for c in [
        "adaptation_percentage", "subset_seed", "training_seed",
        "adaptation_train_count", "adaptation_validation_count", "labelled_images_consumed",
        "roc_auc", "average_precision",
        "f1", "f1_at_adaptation_threshold", "f1_at_baseline_threshold",
        "threshold_adaptation_selected", "threshold_baseline_unchanged",
    ] if c in cells.columns]
    display(cells[columns].sort_values("adaptation_percentage"))

    print("\nacross-seed summary (from the run's own summary block):")
    summary = pd.DataFrame(metrics["summary"]["rows"])
    if not summary.empty:
        display(summary[summary["metric"].isin(["f1", "roc_auc"])])

    # Interpretation guard: if ROC-AUC is already flat while F1 moves, the curve is
    # measuring calibration, not adaptation. State which one the text is claiming.
    if "roc_auc" in cells.columns and cells["roc_auc"].notna().any():
        spread = cells["roc_auc"].max() - cells["roc_auc"].min()
        print(f"\nROC-AUC spread across all budgets: {spread:.4f}")
        if spread < 0.01:
            print("  ROC-AUC is essentially flat -> any F1 movement is a THRESHOLD effect,")
            print("  not evidence that adaptation improved the ranking.")

run_id        : fine_tuning-20260808T144640138049Z-6ec6cf50a2-f38d
environment   : python 3.11.15 torch 2.13.0 transformers 5.14.1 cuda=False
held out          : biggan
adaptation pool   : 96
final test        : 24 (prevalence 0.5)
baseline threshold: 0.31 (selected_on_seen_generator_validation_only__grid_search__no_held_out_samples)
starting ckpt     : {'epoch': 2, 'fine_tune_mode': 'head_only', 'model_name': 'openai/clip-vit-base-patch32', 'seed': 42, 'validation_metric_name': 'f1', 'validation_metric_value': 1.0}


,adaptation_percentage,subset_seed,training_seed,adaptation_train_count,adaptation_validation_count,labelled_images_consumed,roc_auc,average_precision,f1,f1_at_adaptation_threshold,f1_at_baseline_threshold,threshold_adaptation_selected,threshold_baseline_unchanged
0,0.00,NaN,NaN,0,0,0,0.986111,0.988095,0.000000,0.956522,0.956522,NaN,0.31
1,0.05,42.0,42.0,4,2,6,1.000000,1.000000,0.153846,0.923077,0.857143,0.33,0.31
2,0.05,123.0,42.0,4,2,6,1.000000,1.000000,1.000000,0.800000,0.705882,0.35,0.31
3,0.10,42.0,42.0,8,4,12,1.000000,1.000000,0.153846,0.923077,0.857143,0.33,0.31
4,0.10,123.0,42.0,8,4,12,1.000000,1.000000,1.000000,0.800000,0.705882,0.35,0.31
5,0.20,42.0,42.0,16,6,22,1.000000,1.000000,1.000000,0.923077,0.705882,0.39,0.31
6,0.20,123.0,42.0,16,6,22,1.000000,1.000000,1.000000,0.827586,0.705882,0.36,0.31
7,0.50,42.0,42.0,36,12,48,1.000000,1.000000,1.000000,0.923077,0.685714,0.40,0.31
8,0.50,123.0,42.0,36,12,48,1.000000,1.000000,1.000000,0.888889,0.705882,0.37,0.31



across-seed summary (from the run's own summary block):


,adaptation_percentage,fine_tune_mode,labelled_images_consumed,maximum,mean,metric,minimum,recovery_vs_zero_percent,runs,standard_deviation
0,0.05,head_only,[6],1.000000,0.576923,f1,0.153846,0.576923,2,0.598321
2,0.05,head_only,[6],1.000000,1.000000,roc_auc,1.000000,NaN,2,0.000000
4,0.10,head_only,[12],1.000000,0.576923,f1,0.153846,0.576923,2,0.598321
6,0.10,head_only,[12],1.000000,1.000000,roc_auc,1.000000,NaN,2,0.000000
8,0.20,head_only,[22],1.000000,1.000000,f1,1.000000,1.000000,2,0.000000
10,0.20,head_only,[22],1.000000,1.000000,roc_auc,1.000000,NaN,2,0.000000
12,0.50,head_only,[48],1.000000,1.000000,f1,1.000000,1.000000,2,0.000000
14,0.50,head_only,[48],1.000000,1.000000,roc_auc,1.000000,NaN,2,0.000000
16,0.00,none,[0],0.000000,0.000000,f1,0.000000,0.000000,1,0.000000
18,0.00,none,[0],0.986111,0.986111,roc_auc,0.986111,NaN,1,0.000000



ROC-AUC spread across all budgets: 0.0139


## 4. Fine-tuning-depth ablation

### Questions to answer

- Does head-only adaptation recover performance, suggesting that frozen CLIP features already contain useful separation?
- Does unfreezing the last transformer block add consistent benefit for its compute/overfitting cost?
- Does full fine-tuning improve further, or become less stable in limited-data regimes?
- Were starting weights, subset IDs, final-test IDs, training budgets, and selection rules held constant?
- Could differences reflect optimiser/hyperparameter suitability rather than representational necessity?

### Writing TODO

Frame each conclusion at the level supported by the controlled comparison. Underperformance of full fine-tuning does not prove broad features are irrelevant; it may indicate small-data overfitting or optimisation difficulty.

In [5]:
# 4. Fine-tuning-depth ablation.
run, metrics = load("ablation")
if run is None:
    print("No completed ablation run yet (run it on ONE representative held-out generator).")
else:
    provenance(run)
    controls = metrics["controls"]
    print(f"modes            : {', '.join(metrics['modes'])}")
    print(f"budget policy    : {controls['training_budget_policy']}")
    print(f"epoch-budget violations: {controls['budget_policy_violated_by'] or 'none'}")
    print(f"mode overrides   : {controls['mode_overrides_applied'] or 'none'}")
    print(f"shared final test: {controls['final_test_sample_id_count']} samples")
    display(pd.DataFrame(metrics["mode_comparison"]))
    print("\ntrainable parameters per mode:")
    display(pd.DataFrame([
        {"mode": mode, "trainable_tensors": len(names)}
        for mode, names in metrics["trainable_parameter_names_by_mode"].items()
    ]))
    print("\n" + metrics["interpretation_note"])

No completed ablation run yet (run it on ONE representative held-out generator).


## 5. Cross-experiment observations

Create a table separating: (a) directly observed pattern, (b) supporting runs/figures, (c) uncertainty/alternative explanations, and (d) further test that would discriminate explanations. Consider generator heterogeneity, content/domain shift, dataset artefacts, class prevalence, calibration, subset variance, and compute trade-offs.

Do not use causal language for observational differences unless the experimental control genuinely supports it.

In [6]:
# 5. Cross-experiment observations.
#
# Each row must name the run that evidences it, so no claim in the write-up is
# unsourced. Rows are derived from saved metrics, never typed in by hand.
observations = []
run, metrics = load("unseen_generator")
if run is not None:
    gap = metrics["generalisation_gap"]
    roc = gap.get("roc_auc", {})
    f1 = gap.get("f1_at_default_threshold", {})
    observations.append({
        "observation": f"ranking transfer to unseen {metrics['held_out_generator']}",
        "metric": "roc_auc", "value": roc.get("unseen"),
        "reference": roc.get("in_distribution"), "drop": roc.get("absolute_drop"),
        "evidence_run_id": run.name,
    })
    observations.append({
        "observation": f"fixed-threshold transfer to unseen {metrics['held_out_generator']}",
        "metric": "f1@0.5", "value": f1.get("unseen"),
        "reference": f1.get("in_distribution"), "drop": f1.get("absolute_drop"),
        "evidence_run_id": run.name,
    })
run, metrics = load("fine_tuning")
if run is not None:
    cells = pd.read_csv(run / "recovery_cells.csv")
    adapted = cells[cells["adaptation_percentage"] > 0]
    zero = cells[cells["adaptation_percentage"] == 0]
    if not adapted.empty and not zero.empty:
        for column, name in (("f1_at_adaptation_threshold", "f1@adaptation threshold"),
                             ("roc_auc", "roc_auc")):
            if column in cells.columns and cells[column].notna().any():
                observations.append({
                    "observation": "best recovery across budgets",
                    "metric": name, "value": adapted[column].max(),
                    "reference": zero[column].iloc[0],
                    "drop": None, "evidence_run_id": run.name,
                })
if observations:
    display(pd.DataFrame(observations))
else:
    print("No completed runs to draw observations from yet.")

,observation,metric,value,reference,drop,evidence_run_id
0,ranking transfer to unseen biggan,roc_auc,0.986111,1.000000,0.013889,unseen_generator-20260808T144550443491Z-b386ea...
1,fixed-threshold transfer to unseen biggan,f1@0.5,0.000000,1.000000,1.000000,unseen_generator-20260808T144550443491Z-b386ea...
2,best recovery across budgets,f1@adaptation threshold,0.923077,0.956522,NaN,fine_tuning-20260808T144640138049Z-6ec6cf50a2-...
3,best recovery across budgets,roc_auc,1.000000,0.986111,NaN,fine_tuning-20260808T144640138049Z-6ec6cf50a2-...


## 6. Discussion, validity, and limitations

### Internal validity

Discuss leakage audits, checkpoint/threshold selection, partition isolation, repeated seeds, hyperparameter decisions, and whether comparisons held intended controls constant.

### Construct validity

Discuss what 'generalisation' and 'recovery' mean operationally here. A single dataset/generator comparison may combine generator, content, provenance, and compression shifts.

### External validity

Limit claims to studied generators, real-image domains, post-processing conditions, model family, and data budgets. New generator versions and deployment prevalence may differ.

### Statistical conclusion validity

Discuss sample/group dependence, uncertainty method, multiple comparisons, seed count, small generator slices, and undefined metrics.

### Ethical and practical considerations

Discuss false accusations, dataset licences/privacy, detector misuse, model/environmental compute costs, and why performance estimates should not be presented as universal guarantees.

## Implementation checklist

- [ ] Inventory completed runs and verify comparison invariants.
- [ ] Analyse baseline, unseen performance, recovery, and ablation in that order.
- [ ] Report sample counts, all declared seeds, uncertainty, and undefined metrics.
- [ ] Separate measured observations from proposed explanations.
- [ ] Link every numerical statement and figure to auditable run artefacts.
- [ ] Discuss leakage, dataset artefacts, validity limits, ethics, and compute.
- [ ] Do not insert or infer results until experiments are actually complete.